# Causal localization and control — pilot gate and full run

Implements `docs/specs/causal_spec.md` v3. GPU work runs here (Colab Pro, single A100);
analysis and figures run locally.

**`PILOT = True` is the default and it is a gate, not a preview.** The full run is not
launched until P1–P4 and P7 pass (§12.5). The pilot costs ~10–20 A100-minutes against
~34 A100-hours for the full programme, and its P1–P8 table is the evidence package for
the resourcing conversation (§12.6).

**Anti-peeking rule (binding).** Only nuisance parameters (throughput, storage, variance)
and the pass/fail verdicts may inform the confirmatory run. Pilot effect *locations* are
discarded — they must not narrow the confirmatory grid.


## Setup (run cells 1–4 once per session)

In [ ]:
# Cell 1 — Clone or update the package code from GitHub.
# REPO_BRANCH is explicit: the causal work lives on `probing` until it merges,
# and a plain `git pull` on a stale default-branch checkout silently
# reinstalls older package code than the notebook cells expect.
import os
REPO_URL = "https://github.com/JoaoPedroFPK/codenames-interpretability.git"
REPO_DIR = "/content/codenames-interpretability"
REPO_BRANCH = "probing"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} fetch origin {REPO_BRANCH}
    !git -C {REPO_DIR} checkout {REPO_BRANCH}
    !git -C {REPO_DIR} reset --hard origin/{REPO_BRANCH}
else:
    !git clone -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}

!git -C {REPO_DIR} log --oneline -3

In [ ]:
# Cell 2 — Install the package in editable mode.
!pip install -q -e "{REPO_DIR}[lens]" 

In [ ]:
# === Config ===
# MODEL_KEY: which decoder this session runs. One of:
#   "mistral"      -> Mistral-7B-Instruct-v0.2   (prefix: mistral)
#   "qwen"         -> Qwen2.5-7B-Instruct        (prefix: qwen)
#   "qwen_random"  -> random-init Qwen null      (prefix: random_qwen)
MODEL_KEY = "mistral"

# PILOT gates everything. True  -> n=150 on with_social, emits the P1-P8 table.
#                        False -> the confirmatory n=1500 run (§4.2).
# Do NOT set False until the pilot passes for this model.
PILOT = True

SEED = 2026                      # CONTRACT_V1.random_seed; never change it
SCHEME = "counterfactual"        # primary; "noise" is the ROME comparability arm
DRIVE = "/content/drive/MyDrive/TCC"
DATASET = f"{DRIVE}/clue_generation.csv"

SAMPLE_SIZE = 150 if PILOT else 1500
CONDITION = "with_social" if PILOT else "no_social"
OUTPUT_DIR = f"{DRIVE}/{'causal_pilot' if PILOT else 'causal'}_outputs"

print(f"model={MODEL_KEY} pilot={PILOT} n={SAMPLE_SIZE} condition={CONDITION}")
print(f"output -> {OUTPUT_DIR}")

In [ ]:
# Cell 3 — Verify the installed environment matches the pinned set.
!codenames-experiment doctor --model {MODEL_KEY}

In [ ]:
# Cell 4 — Autoreload and mount Drive.
import sys, types, importlib
if "imp" not in sys.modules:
    # Python 3.12 removed `imp`; old IPython autoreload still imports it.
    _imp = types.ModuleType("imp")
    _imp.reload = importlib.reload
    sys.modules["imp"] = _imp

REPO_DIR = "/content/codenames-interpretability"
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

%load_ext autoreload
%autoreload 2

from google.colab import drive
drive.mount("/content/drive")

## Stage 1 — Paired clean/corrupted extraction

Builds the symmetric-counterfactual pair table (§5A) and caches clean and corrupted
per-layer states. Pairs whose prompts differ in token count are dropped and logged:
patching `(layer, position)` across misaligned sequences reads the wrong position.

In [ ]:
!codenames-experiment causal-extract \
    --model {MODEL_KEY} \
    --dataset {DATASET} \
    --output-dir {OUTPUT_DIR} \
    --scheme {SCHEME} \
    --condition {CONDITION} \
    --sample-size {SAMPLE_SIZE} \
    --seed {SEED}

## Stage 2 — Pilot gate (P1–P8)

**Read the table before going further.** If any of P1, P2, P3, P4 or P7 fails, stop and
fix the pipeline — do not run the confirmatory stages. P5 failing costs the attribution
shortcut and forces a re-cost; P6 failing converts RQ2 into a pre-registered bounded
negative (§2.1 row 4) and does not block.

In [ ]:
if PILOT:
    !codenames-experiment causal-pilot \
        --model {MODEL_KEY} \
        --dataset {DATASET} \
        --output-dir {OUTPUT_DIR} \
        --sample-size {SAMPLE_SIZE} \
        --condition {CONDITION} \
        --seed {SEED}
else:
    print("PILOT=False - skipping the gate (it must already have passed).")

## Stage 3 — Attribution screen, real patches, steering

Stage 1 of the two-stage design is screening only and carries no inferential claim
(§3.4). Every locus it surfaces is confirmed with a real patch, and a random 10% of the
grid is patched regardless of score to bound the false-negative rate.

In [ ]:
!codenames-experiment causal-scan \
    --model {MODEL_KEY} --output-dir {OUTPUT_DIR} \
    --scheme {SCHEME} --condition {CONDITION} --seed {SEED}

In [ ]:
!codenames-experiment causal-patch \
    --model {MODEL_KEY} --output-dir {OUTPUT_DIR} \
    --scheme {SCHEME} --condition {CONDITION} \
    --sample-size {SAMPLE_SIZE} --seed {SEED} --resume

In [ ]:
!codenames-experiment causal-steer \
    --model {MODEL_KEY} --output-dir {OUTPUT_DIR} \
    --condition {CONDITION} --direction lens \
    --sample-size {SAMPLE_SIZE} --seed {SEED}

## Next steps (local, no GPU)

Sync `OUTPUT_DIR` down from Drive, then:

```bash
codenames-experiment causal-analyze --model mistral \
    --output-dir output/causal_outputs --condition no_social --figures
```

`causal-analyze` applies the §3.4 claim gate (BH-FDR, cluster bootstrap over turns,
paired contrasts) and writes the patching heatmap, dose–response curves, and the
triangulation figure.